# Statistical Analysis

This notebook reproduces the statistical analyses reported in the study.

Analyses:
1. Fit sigmoid functions to embedding similarity continua.
2. Compare sigmoid slope parameters (k) between models.
3. Evaluate sigmoid goodness-of-fit (MAE).
4. Quantify the Ganong effect in the final transformer layer of fine-tuned wav2vec2.

## Load embedding similarity data

Embedding similarity scores were computed separately for Whisper and
wav2vec2 representations. For each layer, a sigmoid function was fitted
to the continuum responses using the procedures implemented in
`sigmoid_fitting.py`.

In [1]:
from scipy import stats
import pandas as pd
from sigmoid_fitting import fit_single_layer, run_all_layers

In [2]:
data_wav2vec2 = pd.read_csv("../Data/wav2vec2_pretrained_embed_data.csv")
data_whisper = pd.read_csv("../Data/Whisper_embed_data.csv")

fitted_whisper = run_all_layers(data_whisper)
fitted_wav2vec2 = run_all_layers(data_wav2vec2)

## Comparison of sigmoid slopes between models

To assess whether the two speech models differ in category boundarysharpness, I compare layer-wise sigmoid slope parameters (k) using a paired-samples t-test. Each layer contributes one observation from Whisper and one observation from wav2vec2.

In [3]:
diff = fitted_whisper['k'] - fitted_wav2vec2['k']

result = stats.ttest_rel(
    fitted_whisper['k'],
    fitted_wav2vec2['k']
)

ci = result.confidence_interval(confidence_level=0.95)

print("Mean Whisper k:", fitted_whisper['k'].mean())
print("Mean Wav2Vec2 k:", fitted_wav2vec2['k'].mean())
print("Mean diff:", diff.mean())
print("95% CI:", ci)
print("t:", result.statistic)
print("p-val:", result.pvalue)

Mean Whisper k: 0.7185062569908834
Mean Wav2Vec2 k: 0.5864764937143708
Mean diff: 0.13202976327651256
95% CI: ConfidenceInterval(low=np.float64(0.05245190004208443), high=np.float64(0.21160762651094067))
t: 3.6149266948378473
p-val: 0.003547418960207447


## Model fit quality

Mean Absolute Error (MAE) quantifies the difference between observed similarity values and the fitted sigmoid curve.

Lower MAE values indicate better fit.

In [4]:
print("Mean (Layer-wise) Whisper MAE", fitted_whisper['mae'].mean())
print("Mean (Layer-wise) wav2vec2 MAE", fitted_wav2vec2['mae'].mean())

Mean (Layer-wise) Whisper MAE 0.03634841211082483
Mean (Layer-wise) wav2vec2 MAE 0.03194903142028201


## Descriptive statistics for each model ##

In [5]:
fitted_whisper.describe()

,Layer,k,mae
count,13.00000,13.000000,13.000000
mean,6.00000,0.718506,0.036348
std,3.89444,0.097443,0.004244
min,0.00000,0.598555,0.029809
25%,3.00000,0.618787,0.034249
50%,6.00000,0.708437,0.036009
75%,9.00000,0.788998,0.040138
max,12.00000,0.892393,0.043429


In [6]:
fitted_wav2vec2.describe()

,Layer,k,mae
count,13.00000,13.000000,13.000000
mean,6.00000,0.586476,0.031949
std,3.89444,0.156427,0.008389
min,0.00000,0.477663,0.021489
25%,3.00000,0.509164,0.025045
50%,6.00000,0.529230,0.032841
75%,9.00000,0.572659,0.036468
max,12.00000,1.037440,0.050327


## Quantifying the Ganong effect

The Ganong effect is quantified as the difference in sigmoid midpoints ($x_0$) between the two continua.

Sigmoid functions are fitted independently to the final transformer layer representations of the two continua, and the difference in estimated category boundaries is computed.

In [7]:
mask_dash = data_wav2vec2['Layer'].isin([12]) & data_wav2vec2['Pair'].isin(['dash-tash'])
mask_task = data_wav2vec2['Layer'].isin([12]) & data_wav2vec2['Pair'].isin(['task-dask'])

popt, mae = fit_single_layer(data_wav2vec2[mask_dash])
popt2, mae2 = fit_single_layer(data_wav2vec2[mask_task])

ganong_shift = popt[1] - popt2[1]

print(ganong_shift)

0.013921418781737138
